In [2]:

import logging
import time
from functools import wraps

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

logger = logging.getLogger("API")


In [3]:

def timing(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()
        logger.info(f"{func.__name__} took {end - start:.4f} seconds")

        return result

    return wrapper


In [4]:

def log_io(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"Input to {func.__name__}: args={args}, kwargs={kwargs}")

        result = func(*args, **kwargs)

        logger.info(f"Output from {func.__name__}: {result}")

        return result

    return wrapper


In [5]:

def check_role(allowed_roles):
    def decorator(func):
        @wraps(func)
        def wrapper(role, *args, **kwargs):
            if role not in allowed_roles:
                logger.warning(f"Access denied for role: {role}")
                raise PermissionError("You are not allowed to access this API")

            logger.info(f"Access granted for role: {role}")

            return func(role, *args, **kwargs)

        return wrapper
    return decorator


In [6]:

@timing
@log_io
@check_role(["admin", "user"])
def get_order(role, order_id):
    return f"Order {order_id} details"


In [7]:

try:
    result = get_order("admin", 101)
    logger.info(f"API Result: {result}")

except PermissionError as error:
    logger.error(str(error))

try:
    result = get_order("guest", 102)
    logger.info(f"API Result: {result}")

except PermissionError as error:
    logger.error(str(error))


INFO: Input to get_order: args=('admin', 101), kwargs={}
INFO: Access granted for role: admin
INFO: Output from get_order: Order 101 details
INFO: get_order took 0.0030 seconds
INFO: API Result: Order 101 details
INFO: Input to get_order: args=('guest', 102), kwargs={}
ERROR: You are not allowed to access this API
